# Stimulus Classification via Topological Turnover Rate

This notebook performs a full classification analysis using the **TurnoverRate**
vectorization of zigzag persistence diagrams computed from 3-D neural activity
grids ($15 \times 15 \times 10 \times T$).

The TurnoverRate at frame $t$ and homological dimension $d$ is:

$$T_d(t) = \frac{\text{births}_d(t) + \text{deaths}_d(t)}{\beta_d(t-1) + \beta_d(t) + 1}$$

It measures how rapidly the topology of neural activity reorganises at each
time step — a *volatility* measure of the neural topology.

**Outline**

| Section | Description |
|---|---|
| **1** | Setup & data loading |
| **2** | Per-mouse stimulus classification (4-class, within each mouse) |
| **3** | Pooled cross-mouse classification (join mice sharing the same label set) |
| **4** | Interpretability: mean turnover profiles, discriminative frames, PCA |
| **5** | Summary |

## 1. Setup & data loading

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

from zztop.vectorizations import TurnoverRate
from zztop.vectorizations._diagram import normalize_diagram

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT = Path("/orfeo/scratch/area/mbiagetti/project_cc")
META_ROOT = Path("/u/area/mbiagetti/Codes/sensorium_metadata/metadata")

P_ACTIVE  = 30
ZZ_FOLDER = f"trials_zz-thresh-{P_ACTIVE}"

mice = sorted(
    d.name for d in DATA_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("dynamic")
)
print(f"Found {len(mice)} mice with zigzag data")

Found 10 mice with zigzag data


In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)


def load_trial_metadata(mouse_name):
    csv_path = META_ROOT / mouse_name / "trials" / f"meta-trials_{mouse_name}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    return pd.read_csv(csv_path)


def load_labelled_barcodes(mouse_name):
    """Load zigzag barcodes as raw numpy arrays (no tuple conversion)."""
    df = load_trial_metadata(mouse_name)
    trial_to_label  = dict(zip(df["trial"].astype(int), df["label"]))
    trial_to_frames = dict(zip(df["trial"].astype(int), df["valid_frames"].astype(int)))

    zz_dir = DATA_ROOT / mouse_name / ZZ_FOLDER
    files  = sorted(f for f in zz_dir.glob("zz-thresh-*.npy") if "info" not in f.name)

    barcodes, labels_list, frames_list = [], [], []
    for f in files:
        match = re.search(r"trial-(\d+)", f.stem)
        if match is None:
            continue
        tid = int(match.group(1))
        if tid not in trial_to_label:
            continue
        raw = np.load(f, allow_pickle=True)
        barcodes.append(np.asarray(raw, dtype=np.float64))
        labels_list.append(trial_to_label[tid])
        frames_list.append(trial_to_frames[tid])

    return barcodes, np.array(labels_list), np.array(frames_list)


def clip_barcode(arr, n_frames):
    """Clip one barcode array to [0, n_frames) — fully vectorized."""
    mask = arr[:, 1] < n_frames
    arr = arr[mask].copy()
    arr[:, 2] = np.minimum(arr[:, 2], n_frames)
    finite = np.isfinite(arr[:, 2])
    return arr[finite]


def fast_turnover_rate(arr, n_frames, dimensions=(0, 1, 2)):
    """Compute TurnoverRate from a (n_bars, 3) barcode array.

    Uses np.bincount + np.cumsum for O(n_bars + n_frames) per dimension,
    instead of the O(n_bars * n_frames) Python loop in the library.
    """
    profiles = []
    dims_col = arr[:, 0].astype(int) if arr.size > 0 else np.array([], dtype=int)

    for dim in dimensions:
        profile = np.zeros(n_frames, dtype=np.float64)
        if arr.size == 0:
            profiles.append(profile)
            continue

        dim_mask = dims_col == dim
        bd = arr[dim_mask]
        if bd.shape[0] == 0:
            profiles.append(profile)
            continue

        births = np.clip(bd[:, 1].astype(int), 0, n_frames - 1)
        deaths = bd[:, 2].astype(int)  # can be up to n_frames

        # Histogram of births/deaths per frame — O(n_bars)
        births_at = np.bincount(births, minlength=n_frames)[:n_frames].astype(np.float64)
        deaths_at = np.bincount(np.clip(deaths, 0, n_frames - 1),
                                minlength=n_frames)[:n_frames].astype(np.float64)
        # Zero out deaths at or beyond n_frames (they don't die within window)
        # deaths exactly at n_frames land in bin n_frames-1 after clip, undo that:
        deaths_at_fixed = np.zeros(n_frames, dtype=np.float64)
        valid_deaths = deaths[(deaths >= 0) & (deaths < n_frames)]
        if valid_deaths.size > 0:
            deaths_at_fixed += np.bincount(valid_deaths, minlength=n_frames)[:n_frames].astype(np.float64)

        # Betti via cumsum — O(n_frames)
        betti = np.cumsum(births_at) - np.cumsum(deaths_at_fixed)

        prev_betti = np.empty_like(betti)
        prev_betti[0] = 0.0
        prev_betti[1:] = betti[:-1]

        denom = prev_betti + betti + 1.0
        profile = (births_at + deaths_at_fixed) / denom
        profiles.append(profile)

    return np.concatenate(profiles)


def prepare_mouse(mouse_name):
    """Load, clip, and vectorise one mouse. Caches result to disk."""
    cache_file = CACHE_DIR / f"turnover_{mouse_name}.npz"
    if cache_file.exists():
        data = np.load(cache_file, allow_pickle=True)
        return {
            "X": data["X"],
            "labels": data["labels"],
            "clip_frames": int(data["clip_frames"]),
            "n_trials": int(data["n_trials"]),
            "label_set": sorted(set(data["labels"])),
            "mouse": mouse_name,
        }

    barcodes, labels, valid_frames = load_labelled_barcodes(mouse_name)
    clip = int(valid_frames.min())

    # Vectorised turnover rate — ~240x faster than library version
    X = np.zeros((len(barcodes), clip * 3), dtype=np.float64)
    for i, bc in enumerate(barcodes):
        clipped = clip_barcode(bc, clip)
        X[i] = fast_turnover_rate(clipped, clip)
    X = np.nan_to_num(X)

    # Cache to disk for instant re-runs
    np.savez_compressed(cache_file, X=X, labels=labels,
                        clip_frames=clip, n_trials=len(labels))

    return {
        "X": X,
        "labels": labels,
        "clip_frames": clip,
        "n_trials": len(labels),
        "label_set": sorted(set(labels)),
        "mouse": mouse_name,
    }

In [ ]:
# ── Load & vectorise all mice ─────────────────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

t0 = time.perf_counter()
mouse_data = {}

# ThreadPool: avoids pickle overhead, good for I/O-bound file loading
with ThreadPoolExecutor(max_workers=min(len(mice), 6)) as pool:
    futures = {pool.submit(prepare_mouse, m): m for m in mice}
    for fut in as_completed(futures):
        m = futures[fut]
        try:
            d = fut.result()
            mouse_data[m] = d
            elapsed = time.perf_counter() - t0
            print(f"  [{elapsed:5.1f}s] {m[:45]:45s}  trials={d['n_trials']:>4d}  "
                  f"clip={d['clip_frames']:>3d}  labels={d['label_set']}  "
                  f"X={d['X'].shape}")
        except Exception as e:
            import traceback
            print(f"  {m[:45]:45s}  SKIPPED — {e}")
            traceback.print_exc()

elapsed = time.perf_counter() - t0
print(f"\nLoaded {len(mouse_data)} / {len(mice)} mice in {elapsed:.1f}s")
print("(Results cached — re-runs will be instant. Delete cache/ to recompute.)")

  [651.5s] dynamic29623-4-9-Video-9b4f6a1a067fe51e15306b  trials= 420  clip=240  labels=[np.str_('Gabor'), np.str_('NaturalImages'), np.str_('NaturalVideo'), np.str_('RandomDots')]  X=(420, 720)
  [676.0s] dynamic29515-10-12-Video-9b4f6a1a067fe51e1530  trials= 445  clip=240  labels=[np.str_('GaussianDot'), np.str_('NaturalVideo'), np.str_('PinkNoise'), np.str_('RandomDots')]  X=(445, 720)


KeyboardInterrupt: 

## 2. Per-mouse stimulus classification

For each mouse independently, run 5-fold stratified CV with
logistic regression on the TurnoverRate features.  Each mouse has
4 stimulus classes (NaturalVideo + 3 others).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"acc": "accuracy", "f1": "f1_macro"}

per_mouse_results = {}

for mname, md in mouse_data.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, random_state=42, class_weight="balanced",
        )),
    ])
    cv_res = cross_validate(pipe, md["X"], md["labels"], cv=cv,
                            scoring=scoring)
    res = {
        "acc":  cv_res["test_acc"].mean(),
        "acc_std": cv_res["test_acc"].std(),
        "f1":   cv_res["test_f1"].mean(),
        "f1_std":  cv_res["test_f1"].std(),
        "n_trials": md["n_trials"],
        "labels": md["label_set"],
        "clip":  md["clip_frames"],
    }
    per_mouse_results[mname] = res
    short = mname.split("-")[0].replace("dynamic", "")
    print(f"  Mouse {short}: acc={res['acc']:.3f}±{res['acc_std']:.3f}  "
          f"F1={res['f1']:.3f}±{res['f1_std']:.3f}  "
          f"(n={res['n_trials']}, clip={res['clip']}, {len(res['labels'])} classes)")

In [ ]:
# ── Bar chart: per-mouse accuracy & F1 ──────────────────────────────────────
short_names = [m.split("-")[0].replace("dynamic", "") for m in per_mouse_results]
accs  = [per_mouse_results[m]["acc"]     for m in per_mouse_results]
a_err = [per_mouse_results[m]["acc_std"] for m in per_mouse_results]
f1s   = [per_mouse_results[m]["f1"]      for m in per_mouse_results]
f_err = [per_mouse_results[m]["f1_std"]  for m in per_mouse_results]

x = np.arange(len(short_names))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, accs, w, yerr=a_err, capsize=3, alpha=0.8, label="Accuracy")
ax.bar(x + w/2, f1s,  w, yerr=f_err, capsize=3, alpha=0.5,
       hatch="//", label="Macro F1")
ax.axhline(0.25, color="gray", ls="--", alpha=0.5, label="Chance (1/4)")
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_xlabel("Mouse ID")
ax.set_ylabel("Score")
ax.set_title("Per-mouse stimulus classification — TurnoverRate + LogReg", fontsize=13)
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-mouse confusion matrices ────────────────────────────────────────────
n_mice = len(mouse_data)
n_cols = min(5, n_mice)
n_rows = (n_mice + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_mice == 1:
    axes = np.array([axes])
axes_flat = axes.flat

for ax, (mname, md) in zip(axes_flat, mouse_data.items()):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, random_state=42, class_weight="balanced",
        )),
    ])
    y_pred = cross_val_predict(pipe, md["X"], md["labels"], cv=cv)
    classes = sorted(set(md["labels"]))
    # Short class names for readability
    short_cls = [c[:6] for c in classes]
    cm = confusion_matrix(md["labels"], y_pred, labels=classes)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    ConfusionMatrixDisplay(cm_norm, display_labels=short_cls).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    short = mname.split("-")[0].replace("dynamic", "")
    f1 = per_mouse_results[mname]["f1"]
    ax.set_title(f"Mouse {short}\nF1={f1:.3f}", fontsize=10)

for ax in list(axes_flat)[n_mice:]:
    ax.set_visible(False)

fig.suptitle("Normalised confusion matrices (per-mouse, TurnoverRate)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Cross-mouse (pooled) stimulus classification

Pool trials from **all mice that share the same stimulus set** and train a
single classifier. This tests whether the topological volatility signature
of each stimulus generalises across different neural circuits.

Because different mice have different clip lengths, we re-clip everything to
a **global minimum** across all mice so vectors are the same dimensionality.

In [ ]:
# ── Group mice by stimulus set ───────────────────────────────────────────────
groups = defaultdict(list)
for mname, md in mouse_data.items():
    key = tuple(md["label_set"])
    groups[key].append(mname)

print("Stimulus-set groups:")
for labels, members in groups.items():
    ids = [m.split('-')[0].replace('dynamic','') for m in members]
    print(f"  {list(labels)} → {len(members)} mice: {ids}")

In [ ]:
# ── Cross-mouse classification per group ─────────────────────────────────────
cross_results = {}

for label_key, members in groups.items():
    if len(members) < 2:
        print(f"Skipping {list(label_key)} — only 1 mouse")
        continue

    # Global clip = min across all mice in this group
    global_clip = min(mouse_data[m]["clip_frames"] for m in members)
    print(f"\nGroup {list(label_key)} — {len(members)} mice, global_clip={global_clip}")

    # Re-vectorise at global_clip using cached barcodes or re-loading
    all_X, all_labels, all_mouse_ids = [], [], []
    for mname in members:
        md = mouse_data[mname]
        own_clip = md["clip_frames"]
        if own_clip == global_clip:
            # Same clip — reuse pre-computed X
            all_X.append(md["X"])
            all_labels.append(md["labels"])
        else:
            # Need to re-clip at global_clip (truncate feature vector)
            # TurnoverRate output spans [0..clip) per dim — just take first global_clip per dim
            n_dim = md["X"].shape[1] // own_clip
            cols = np.concatenate([
                np.arange(d * own_clip, d * own_clip + global_clip)
                for d in range(n_dim)
            ])
            all_X.append(md["X"][:, cols])
            all_labels.append(md["labels"])
        all_mouse_ids.extend([mname] * len(md["labels"]))
        short = mname.split('-')[0].replace('dynamic','')
        print(f"  Mouse {short}: {md['n_trials']} trials")

    X_pool = np.vstack(all_X)
    y_pool = np.concatenate(all_labels)
    mouse_ids = np.array(all_mouse_ids)
    print(f"  Pooled: {X_pool.shape[0]} trials, {X_pool.shape[1]} features")

    # --- Standard CV (trials shuffled across mice) ---
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, random_state=42, class_weight="balanced",
        )),
    ])
    cv_res = cross_validate(pipe, X_pool, y_pool, cv=cv, scoring=scoring)
    acc = cv_res["test_acc"].mean()
    f1  = cv_res["test_f1"].mean()
    print(f"  Pooled CV  → acc={acc:.3f}  F1={f1:.3f}")

    # --- Leave-one-mouse-out CV ---
    from sklearn.model_selection import LeaveOneGroupOut
    logo = LeaveOneGroupOut()
    logo_res = cross_validate(pipe, X_pool, y_pool,
                              cv=logo, groups=mouse_ids, scoring=scoring)
    logo_acc = logo_res["test_acc"].mean()
    logo_f1  = logo_res["test_f1"].mean()
    print(f"  Leave-1-mouse-out → acc={logo_acc:.3f}  F1={logo_f1:.3f}")

    cross_results[label_key] = {
        "pooled_acc": acc, "pooled_f1": f1,
        "logo_acc": logo_acc, "logo_f1": logo_f1,
        "n_trials": len(y_pool), "n_mice": len(members),
        "global_clip": global_clip,
        "X_pool": X_pool, "y_pool": y_pool, "mouse_ids": mouse_ids,
    }

In [ ]:
# ── Confusion matrix for each pooled group (leave-one-mouse-out) ─────────────
for label_key, cr in cross_results.items():
    from sklearn.model_selection import LeaveOneGroupOut
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, random_state=42, class_weight="balanced",
        )),
    ])
    logo = LeaveOneGroupOut()
    y_pred = cross_val_predict(pipe, cr["X_pool"], cr["y_pool"],
                               cv=logo, groups=cr["mouse_ids"])

    classes = sorted(set(cr["y_pool"]))
    short_cls = [c[:8] for c in classes]
    cm = confusion_matrix(cr["y_pool"], y_pred, labels=classes)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    ConfusionMatrixDisplay(cm, display_labels=short_cls).plot(
        ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title("Counts")
    ConfusionMatrixDisplay(cm_norm, display_labels=short_cls).plot(
        ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
    axes[1].set_title("Recall")

    fig.suptitle(
        f"Leave-one-mouse-out confusion — {list(label_key)}\n"
        f"{cr['n_mice']} mice, {cr['n_trials']} trials, "
        f"F1={cr['logo_f1']:.3f}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

## 4. Interpretability

To verify the classifier is learning meaningful topological differences
(not artefacts), we examine:

1. **Mean turnover profiles** per stimulus — do they look different?
2. **LogReg coefficients** — which frames/dimensions drive the decision?
3. **PCA of turnover vectors** — do stimuli form clusters?
4. **Dimensional contributions** — which homological dimension (0, 1, 2) matters most?

In [ ]:
# ── 4a. Mean turnover profiles per stimulus (one reference mouse) ────────────
# Pick the mouse with the highest per-mouse F1 for illustration
ref_mouse = max(per_mouse_results, key=lambda m: per_mouse_results[m]["f1"])
ref = mouse_data[ref_mouse]
ref_short = ref_mouse.split("-")[0].replace("dynamic", "")

clip = ref["clip_frames"]
n_dim = ref["X"].shape[1] // clip   # number of homological dimensions
dim_labels = [f"H{d}" for d in range(n_dim)]

fig, axes = plt.subplots(1, n_dim, figsize=(6 * n_dim, 4), sharey=True)
if n_dim == 1:
    axes = [axes]

for d_idx, ax in enumerate(axes):
    start, end = d_idx * clip, (d_idx + 1) * clip
    for label in sorted(set(ref["labels"])):
        mask = ref["labels"] == label
        mean_prof = ref["X"][mask, start:end].mean(axis=0)
        std_prof  = ref["X"][mask, start:end].std(axis=0)
        ax.plot(mean_prof, label=label, linewidth=1.2)
        ax.fill_between(range(clip), mean_prof - std_prof,
                        mean_prof + std_prof, alpha=0.15)
    ax.set_title(f"{dim_labels[d_idx]} (dim {d_idx})", fontsize=11)
    ax.set_xlabel("Frame")
    if d_idx == 0:
        ax.set_ylabel("Turnover rate")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(alpha=0.3)

fig.suptitle(
    f"Mean turnover profiles by stimulus — Mouse {ref_short}\n"
    f"(shaded = ±1 std across trials)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4b. Logistic regression coefficients — discriminative frames ─────────────
pipe_interp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000, random_state=42, class_weight="balanced",
    )),
])
pipe_interp.fit(ref["X"], ref["labels"])

coefs = pipe_interp.named_steps["clf"].coef_   # (n_classes, n_features)
classes = pipe_interp.named_steps["clf"].classes_

fig, axes = plt.subplots(len(classes), 1, figsize=(14, 3 * len(classes)),
                          sharex=True)
if len(classes) == 1:
    axes = [axes]

for i, (cls_name, ax) in enumerate(zip(classes, axes)):
    c = coefs[i]
    for d_idx in range(n_dim):
        start, end = d_idx * clip, (d_idx + 1) * clip
        ax.plot(range(start, end), c[start:end],
                label=dim_labels[d_idx], linewidth=0.9)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_ylabel(f"{cls_name}\ncoefficient")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Feature index (frame within each dimension)")
fig.suptitle(
    f"Logistic regression coefficients per class — Mouse {ref_short}\n"
    f"(positive = evidence FOR that class)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4c. PCA of turnover vectors coloured by stimulus ─────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(ref["X"])
X_scaled = np.nan_to_num(X_scaled)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 vs PC2
for label in sorted(set(ref["labels"])):
    mask = ref["labels"] == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.5, label=label)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[0].set_title("PC1 vs PC2")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# PC1 vs PC3
for label in sorted(set(ref["labels"])):
    mask = ref["labels"] == label
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 2], s=8, alpha=0.5, label=label)
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC3 ({pca.explained_variance_ratio_[2]:.1%})")
axes[1].set_title("PC1 vs PC3")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.suptitle(
    f"PCA of TurnoverRate vectors — Mouse {ref_short}\n"
    f"Total variance explained (3 PCs): {pca.explained_variance_ratio_.sum():.1%}",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4d. Dimensional contribution — classify using each H_d alone ─────────────
dim_results = {}

for d_idx in range(n_dim):
    start, end = d_idx * clip, (d_idx + 1) * clip
    X_dim = ref["X"][:, start:end]
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, random_state=42, class_weight="balanced",
        )),
    ])
    cv_res = cross_validate(pipe, X_dim, ref["labels"], cv=cv, scoring=scoring)
    dim_results[d_idx] = {
        "acc": cv_res["test_acc"].mean(),
        "f1":  cv_res["test_f1"].mean(),
    }
    print(f"  H{d_idx} only: acc={dim_results[d_idx]['acc']:.3f}  "
          f"F1={dim_results[d_idx]['f1']:.3f}")

# Also report the combined result
ref_f1 = per_mouse_results[ref_mouse]["f1"]
print(f"  All dims: F1={ref_f1:.3f}")

# Bar chart
dims_list = list(dim_results.keys())
f1s = [dim_results[d]["f1"] for d in dims_list] + [ref_f1]
bar_labels = [f"H{d}" for d in dims_list] + ["All"]

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["tab:blue"] * len(dims_list) + ["tab:green"]
ax.bar(bar_labels, f1s, color=colors, alpha=0.8)
ax.axhline(0.25, color="gray", ls="--", alpha=0.5, label="Chance")
ax.set_ylabel("Macro F1")
ax.set_title(f"Contribution of each homological dimension — Mouse {ref_short}",
             fontsize=12)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4e. Top discriminative frames — absolute coefficient magnitude ───────────
abs_coefs = np.abs(coefs).mean(axis=0)  # mean across classes

fig, ax = plt.subplots(figsize=(14, 4))
for d_idx in range(n_dim):
    start, end = d_idx * clip, (d_idx + 1) * clip
    frames = np.arange(start, end)
    ax.fill_between(frames, abs_coefs[start:end], alpha=0.4,
                    label=dim_labels[d_idx])
    ax.plot(frames, abs_coefs[start:end], linewidth=0.8)

ax.set_xlabel("Feature index (frame within dimension block)")
ax.set_ylabel("|coefficient| (mean across classes)")
ax.set_title(
    f"Discriminative importance of each frame — Mouse {ref_short}\n"
    f"Higher = more important for distinguishing stimuli",
    fontsize=12,
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Report the top-10 most discriminative frames per dimension
print("\nTop-10 most discriminative frames per dimension:")
for d_idx in range(n_dim):
    start = d_idx * clip
    dim_coefs = abs_coefs[start:start + clip]
    top10 = np.argsort(dim_coefs)[::-1][:10]
    print(f"  H{d_idx}: frames {top10.tolist()}  "
          f"(|coef| = {dim_coefs[top10].round(3).tolist()})")

## 5. Summary

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print("=" * 80)
print("PER-MOUSE CLASSIFICATION (TurnoverRate + LogReg, 5-fold CV)")
print("=" * 80)
print(f"{'Mouse':>8s} | {'Trials':>6s} | {'Clip':>4s} | {'Labels':>4s} | "
      f"{'Acc':>8s} | {'F1':>8s}")
print("-" * 56)
for mname, res in per_mouse_results.items():
    short = mname.split('-')[0].replace('dynamic','')
    print(f"{short:>8s} | {res['n_trials']:>6d} | {res['clip']:>4d} | "
          f"{len(res['labels']):>4d} | "
          f"{res['acc']:.3f}±{res['acc_std']:.3f} | "
          f"{res['f1']:.3f}±{res['f1_std']:.3f}")

mean_f1 = np.mean([r["f1"] for r in per_mouse_results.values()])
print(f"\nMean F1 across mice: {mean_f1:.3f}")

if cross_results:
    print("\n" + "=" * 80)
    print("CROSS-MOUSE CLASSIFICATION (pooled & leave-one-mouse-out)")
    print("=" * 80)
    for label_key, cr in cross_results.items():
        print(f"\nStimulus set: {list(label_key)}")
        print(f"  Mice: {cr['n_mice']}, Trials: {cr['n_trials']}, "
              f"Global clip: {cr['global_clip']}")
        print(f"  Pooled CV:          acc={cr['pooled_acc']:.3f}  F1={cr['pooled_f1']:.3f}")
        print(f"  Leave-1-mouse-out:  acc={cr['logo_acc']:.3f}  F1={cr['logo_f1']:.3f}")

print("\n" + "=" * 80)
print("DIMENSION CONTRIBUTION (reference mouse)")
print("=" * 80)
for d_idx, dr in dim_results.items():
    print(f"  H{d_idx}: F1={dr['f1']:.3f}")
print(f"  Combined: F1={ref_f1:.3f}")